# OpenMIC classifier training (Colab)

**Before you run this notebook**

1. Runtime → Change runtime type → **GPU** (L4/T4).
2. Put OpenMIC on Google Drive in this layout:

```text
MyDrive/Decomposition/data/openmic/
├── openmic-2018/
│   ├── audio/
│   ├── openmic-2018.npz
│   ├── class-map.json
│   └── partitions/split01_train.csv, split01_test.csv
└── mel_cache/          # empty is fine; this notebook fills it
```

3. Push the latest repo (with CUDA `train.py`) to GitHub, or Colab will clone stale code.

Source data: [OpenMIC-2018 on Zenodo](https://zenodo.org/record/1432913)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Clone repo

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/alexnguyen25/Decomposition.git"
REPO_DIR = Path("/content/Decomposition")

if REPO_DIR.exists():
    %cd /content/Decomposition
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd /content/Decomposition

sys.path.insert(0, str(REPO_DIR))
print("cwd:", os.getcwd())

## 3. Install training deps

Skip Demucs / Essentia — not needed for the classifier path. Colab already ships a CUDA-enabled PyTorch.

In [ ]:
!pip install -q "librosa>=0.10.0" "soundfile>=0.12.1" "numpy>=1.26,<2"

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 4. Point paths at Drive

Edit `DRIVE_ROOT` if your folder layout differs.

In [ ]:
from pathlib import Path

import src.config as config
import src.classification.train as train_mod
import precompute_mel as precompute_mod

# --- edit this if your Drive layout differs ---
DRIVE_ROOT = Path("/content/drive/MyDrive/Decomposition")
OPENMIC_DIR = DRIVE_ROOT / "data" / "openmic" / "openmic-2018"
CACHE_DIR = DRIVE_ROOT / "data" / "openmic" / "mel_cache"
CHECKPOINT_PATH = DRIVE_ROOT / "models" / "classifier.pt"

assert OPENMIC_DIR.is_dir(), f"Missing OpenMIC dir: {OPENMIC_DIR}"
assert (OPENMIC_DIR / "openmic-2018.npz").exists(), "Missing openmic-2018.npz"
assert (OPENMIC_DIR / "partitions" / "split01_train.csv").exists(), "Missing split01_train.csv"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Override module-level paths used by train + precompute
config.OPENMIC_DIR = OPENMIC_DIR
config.CACHE_DIR = CACHE_DIR

train_mod.OPENMIC_DIR = OPENMIC_DIR
train_mod.CACHE_DIR = CACHE_DIR
train_mod.CHECKPOINT_PATH = CHECKPOINT_PATH

precompute_mod.OPENMIC_DIR = OPENMIC_DIR
precompute_mod.CACHE_DIR = CACHE_DIR

print("OPENMIC_DIR:", OPENMIC_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("Cached mels so far:", len(list(CACHE_DIR.glob("*.npy"))))

## 5. Precompute mel cache (skip if already done)

First run can take a while. Re-runs skip existing `.npy` files.

In [ ]:
precompute_mod.main()
print("Cached mels:", len(list(CACHE_DIR.glob("*.npy"))))

## 6. Train

In [ ]:
train_mod.main()

print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("Checkpoint path:", CHECKPOINT_PATH)
if CHECKPOINT_PATH.exists():
    print("Size (MB):", round(CHECKPOINT_PATH.stat().st_size / 1e6, 3))

## Done

Weights are on Drive at `Decomposition/models/classifier.pt`. Next session you can skip precompute if `mel_cache/` is already populated.